# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arsal626/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Get Hugging Face token safely
def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return getpass.getpass("Enter Hugging Face READ token: ")

hf_token = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Store token in DuckDB session variable rather than putting it in SQL text
con.execute("SET VARIABLE hf_token = ?", [hf_token])
con.execute("""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN getvariable('hf_token'))
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Label window: March 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row in my final modeling frame represents one content item for one client
(`client_hash_id × content_hash_id`). The source warehouse table is daily,
so I aggregate the daily records into a content-client snapshot.

### Decision cutoff

The decision moment is the end of February 2026 (2026-02-28).

### Feature window

February 2026 (2026-02-01 to 2026-02-28) is used to build features. These are
the signals that would have been available at the decision moment.

### Label window

March 2026 (2026-03-01 to 2026-03-31) is kept separate as the future outcome
window. I use it to define whether a content item went dark.

### Prediction goal

The lane ranks content items that may need attention by predicting whether
they will record zero GSC clicks during the following month.

### Deliberately excluded

I exclude future March performance from the feature set because it is only
known after the decision moment and would cause target leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
### Feature fields

The five features I will use are:

1. `impressions_feb` — total GSC impressions during February.
2. `clicks_feb` — total GSC clicks during February.
3. `ctr_feb` — February clicks divided by February impressions.
4. `avg_position_feb` — impression-weighted average GSC position during February.
5. `measured_days_feb` — number of February days where GSC data was available.

Each feature is calculated only from information available by the February 28
decision cutoff.

### Label

`went_dark` — 1 when the content item records zero GSC clicks during March
2026, otherwise 0.

The label is an outcome and is never used as an input feature.

### Context

`client_hash_id` and `content_hash_id` are context fields used to identify,
group, join, and validate rows. They are not model features.

### Excluded

March performance fields are excluded from the feature set because they occur
after the February decision moment. Any field derived from `went_dark` or
March performance is also excluded because it contains future/label information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {FEB}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 10
""").df()

print("Duplicate rows at client × content × date grain:")
print(grain_check)

print("\nNumber of duplicate groups:", len(grain_check))


Duplicate rows at client × content × date grain:
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []

Number of duplicate groups: 0


In [4]:
window_check = con.sql(f"""
SELECT
    'February 2026' AS window,
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {FEB}

UNION ALL

SELECT
    'March 2026' AS window,
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {MAR}
""").df()

display(window_check)

,window,rows,clients,contents,min_date,max_date
0,February 2026,7355108,54,321546,2026-02-01,2026-02-28
1,March 2026,9841378,55,331437,2026-03-01,2026-03-31


In [5]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS unavailable_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) / COUNT(*),
        2
    ) AS available_pct
FROM {FEB}
""").df()

display(availability_check)

,total_rows,available_rows,unavailable_rows,available_pct
0,7355108,2621783,4733325,35.65


In [6]:
feature_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        SUM(gsc_sum_position) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS sum_position_feb,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_days_feb

    FROM {FEB}
    GROUP BY 1, 2
)

SELECT
    client_hash_id,
    content_hash_id,

    impressions_feb,
    clicks_feb,

    clicks_feb / NULLIF(impressions_feb, 0) AS ctr_feb,

    sum_position_feb / NULLIF(impressions_feb, 0)
        AS avg_position_feb,

    measured_days_feb

FROM feb

WHERE impressions_feb >= 100
  AND clicks_feb >= 3
  AND measured_days_feb > 0
""").df()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

Feature frame shape: (29729, 7)


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,measured_days_feb
0,client_3ffa76342f366962,content_5573434837db89c5,198.0,6.0,0.030303,7.318182,24
1,client_3ffa76342f366962,content_7b17975c58745266,102.0,5.0,0.049020,4.450980,26
2,client_3ffa76342f366962,content_b89167cd03d6ffc1,178.0,6.0,0.033708,3.393258,26
3,client_e547b89c05043229,content_eb42160708b4da7c,4802.0,5.0,0.001041,9.917118,28
4,client_e547b89c05043229,content_4a630add28e014a5,7842.0,8.0,0.001020,19.141801,28


### When are the five features available?

| Feature | Available when? |
|---|---|
| `impressions_feb` | Knowable by 2026-02-28 because it uses only measured GSC impressions from February. |
| `clicks_feb` | Knowable by 2026-02-28 because it uses only February GSC clicks. |
| `ctr_feb` | Knowable by 2026-02-28 because it is calculated from February clicks and impressions. |
| `avg_position_feb` | Knowable by 2026-02-28 because it is calculated from February GSC position data. |
| `measured_days_feb` | Knowable by 2026-02-28 because it counts which February days had available GSC data. |

In [7]:
label_frame = con.sql(f"""
WITH mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_mar,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_mar,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_days_mar

    FROM {MAR}
    GROUP BY 1, 2
)

SELECT
    client_hash_id,
    content_hash_id,
    clicks_mar,
    impressions_mar,
    measured_days_mar,

    CASE
        WHEN clicks_mar = 0 THEN 1
        ELSE 0
    END AS went_dark

FROM mar

WHERE measured_days_mar > 0
""").df()

display(label_frame.head())

print("Label distribution:")
print(label_frame["went_dark"].value_counts())

,client_hash_id,content_hash_id,clicks_mar,impressions_mar,measured_days_mar,went_dark
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,0.0,77.0,24,1
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,4.0,602.0,29,0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,1.0,810.0,29,0
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,0.0,82.0,27,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,6.0,1858.0,30,0


Label distribution:
went_dark
1    107901
0     68837
Name: count, dtype: int64


In [8]:
frame = feature_frame.merge(
    label_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "went_dark"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Final frame:", frame.shape)
display(frame.head())

Final frame: (29368, 8)


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,measured_days_feb,went_dark
0,client_3ffa76342f366962,content_5573434837db89c5,198.0,6.0,0.030303,7.318182,24,0
1,client_3ffa76342f366962,content_7b17975c58745266,102.0,5.0,0.049020,4.450980,26,0
2,client_3ffa76342f366962,content_b89167cd03d6ffc1,178.0,6.0,0.033708,3.393258,26,0
3,client_e547b89c05043229,content_eb42160708b4da7c,4802.0,5.0,0.001041,9.917118,28,0
4,client_e547b89c05043229,content_4a630add28e014a5,7842.0,8.0,0.001020,19.141801,28,0


In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# DELIBERATE LEAKAGE:
# Copy the future label directly into a feature.
frame["leaked_label"] = frame["went_dark"]

X_leak = frame[[
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "measured_days_feb",
    "leaked_label"
]]

y = frame["went_dark"]

model_leak = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model_leak.fit(X_leak, y)

pred_leak = model_leak.predict(X_leak)

leak_score = accuracy_score(y, pred_leak)

print(f"Accuracy with leaked feature: {leak_score:.3f}")

Accuracy with leaked feature: 1.000


In [12]:
honest_features = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "measured_days_feb"
]

X_honest = frame[honest_features]

model_honest = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model_honest.fit(X_honest, y)

pred_honest = model_honest.predict(X_honest)

honest_score = accuracy_score(y, pred_honest)

print(f"Accuracy with honest features: {honest_score:.3f}")

Accuracy with honest features: 0.960


In [13]:
frame = frame.drop(columns=["leaked_label"])

print("Leaked feature removed.")
print("Final features:")
print(honest_features)

Leaked feature removed.
Final features:
['impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'measured_days_feb']


### Leakage lesson

I deliberately added `leaked_label`, which was directly derived from the future
label `went_dark`. The model achieved a perfect accuracy of 1.000 because it
was given information that was effectively the answer it was supposed to
predict.

After removing `leaked_label`, the model achieved an accuracy of 0.960 using
only the five February features. This is the honest result for this quick
demonstration.

The key lesson is that label-derived or future-outcome information must never
be included as a model feature. Otherwise, evaluation can look unrealistically
strong even though the model would not have access to that information at the
actual decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

One limitation of this slice is that GSC data availability is not uniform
across all clients and content items. An unavailable measurement period cannot
automatically be interpreted as zero clicks or zero impressions.

Another limitation is that the label requires usable GSC data during the
March outcome window. Therefore, content without measured March data should
not be treated as evidence that the content went dark.

Finally, this experiment uses one February-to-March decision window. Results
from this single period may not generalize to other months, clients, or
seasonal conditions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.